# Evaluating Model trained to regress $M_c$ and $\sigma_{M_c}$

- Dataset: SNR Uniform $\in [15, 30]$
- 150,000 total samples split in 80-10-10
- Original waveform: 512 Hz, 0s to 64s, with merger occuring at 63s.
- Training window: 256 Hz (downsample factor of 2), 0s to 55s.

In [ ]:
import torch
import pandas as pd
import yaml
from BNSReg.core.config import BNSDataModuleRegressionConfig
from BNSReg.tasks.parameter_estimation.model_s4d_gaussnll import LitModelS4DGaussianNLLLoss
from BNSReg.dataloader.regression_loader import LitBNSDataRegression

In [ ]:
ckpt_path = '/n/holystore01/LABS/iaifi_lab/Lab/kyoon/BNSReg/outputs/s4d_gaussnll_snr_15_30_seq_0_55s/setting1-1/checkpoints/s4d_gaussnll_ckpt_epoch=27.ckpt'
config_path = '/n/holystore01/LABS/iaifi_lab/Lab/kyoon/BNSReg/configs/parameter_estimation/user/mc/s4d_gaussnll_15_30_seq_0_55s/s4d_gaussnll_15_30_seq_0_55s_setting1.yaml'
csv_path = '/n/holystore01/LABS/iaifi_lab/Lab/kyoon/BNSReg/outputs/s4d_gaussnll_snr_15_30_seq_0_55s/setting1-1/' + 'out.csv'

In [32]:
# Load YAML
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Extract model init args
model_cfg = cfg['model']['init_args']['model_cfg']
data_cfg = cfg['data']['init_args']['data_cfg']

# Dialing down workers for ease of memory
data_cfg['num_workers'] = 1
data_cfg['prefetch_factor'] = 1
data_cfg['persistent_workers'] = True

In [33]:
model = LitModelS4DGaussianNLLLoss.load_from_checkpoint(
    ckpt_path,
    model_cfg=model_cfg,
    map_location='cpu'
)

/n/home04/kyoon/miniforge3/envs/ssm_cuda312/lib/python3.12/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [34]:
datamodule = LitBNSDataRegression(data_cfg=BNSDataModuleRegressionConfig(**data_cfg))
datamodule.setup(stage='test')
test_loader = datamodule.test_dataloader()

In [ ]:
model.eval()

out = torch.empty(len(test_loader.dataset), 3)
i = 0

with torch.no_grad():
    for batch in test_loader:
        X, y, z = batch # (B, d_input, L), (B, 1), None
        X = X.transpose(2, 1) # (B, d_input, L) -> (B, L, d_input)
        outputs = model(X)
        pred_mean = outputs[:,:1]
        pred_var = model.var_activation(outputs[:,1:])
        
        B = y.size(0)
        out[i;i+B, 0] = y.squeeze(1)
        out[i;i+B, 1] = pred_mean.squeeze(1)
        out[i;i+B, 2] = pred_var.squeeze(1)
        

In [ ]:
out = out.cpu().numpy()
df = pd.DataFrame(
    out,
    columns=['M_chirp_true', 'M_chirp_pred', 'sigma_M_chirp_pred']
)
df.to_csv(csv_path, index=False)